这是**回归分析预测模型 (Regression Analysis)** 的详细解析。

这是数学建模中最**基础**、最**经典**、也是写论文时**必须要有**的模型。
哪怕你后面用了神经网络等高级算法，通常也需要先用回归分析做一个“基准模型 (Baseline)”，或者用它来分析变量之间的**因果关系**（比如：广告费投入对销售额到底有多大影响？）。

---

### 一、 算法原理
**核心思想**：**“透过现象看本质，寻找变量间的数学公式。”**

1.  **函数关系**：假设因变量 $Y$（我们要预测的，如房价）和自变量 $X$（影响因素，如面积、位置）之间存在某种数学关系 $Y = f(X)$。
2.  **最小二乘法 (OLS)**：现实中的点肯定不会乖乖地落在一条线上。我们的目标是画一条线，使得**所有点到这条线的距离（残差）的平方和最小**。
3.  **解释性**：回归分析不仅能预测，还能告诉你**权重**。例如 $y = 0.8x_1 + 0.2x_2$，说明 $x_1$ 的重要性是 $x_2$ 的4倍。

---

### 二、 推导与步骤

#### 1. 确定模型形式
*   **一元线性回归**：$y = \beta_0 + \beta_1 x + \epsilon$ （最简单，画直线）。
*   **多元线性回归**：$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n + \epsilon$ （最常用）。
*   **非线性回归**：$y = ax^2 + bx + c$ 或 $y = a e^{bx}$ （如果数据是弯的）。

#### 2. 估计参数 ($\beta$)
使用**最小二乘法**求解。
$$ \hat{\beta} = (X^T X)^{-1} X^T Y $$
*(不用背公式，Python 会自动算)*

#### 3. 模型检验 (论文得分点)
算出来公式后，不能直接用，必须检验：
*   **拟合优度 ($R^2$)**：越接近1越好。$0.8$以上说明拟合不错。
*   **F检验**：看模型整体是否有意义（线性关系是否显著）。
*   **t检验 (P值)**：看**每一个自变量**是否显著。如果某一项的 $P > 0.05$，说明这一项对结果没啥影响，应该删掉。

#### 4. 预测
代入新的 $X$ 值，算出 $\hat{Y}$。

---

### 三、 适用场景
1.  **因果分析**：分析各因素对结果的影响程度（如：温度、湿度、降雨量对农作物产量的影响）。
2.  **趋势外推**：虽然不如时间序列模型专业，但简单的线性趋势也可以用回归。
3.  **多变量预测**：当你有多个特征（Features）时，回归是首选。

---

### 四、 Python 代码框架

这里强烈推荐使用 `statsmodels` 库，而不是 `sklearn`。
**原因**：`statsmodels` 会打印出一张极其详细的**统计报表**（包含 P值、t值、F值、R方等），这些数据是你要**填进论文表格**里的关键证据。

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import PolynomialFeatures

# 解决中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

class RegressionPredictor:
    def __init__(self, polynomial_degree=1):
        """
        :param polynomial_degree: 多项式阶数。
                                  1 = 线性回归 (y = ax + b)
                                  2 = 二次回归 (y = ax^2 + bx + c)
        """
        self.degree = polynomial_degree
        self.model = None
        self.results = None
        self.poly = None
        
    def fit(self, X, y):
        """
        训练模型
        :param X: 自变量 (n_samples, n_features) 或 (n_samples,)
        :param y: 因变量 (n_samples,)
        """
        # 整理 X 的形状
        X_in = np.array(X)
        if X_in.ndim == 1:
            X_in = X_in.reshape(-1, 1)
            
        y_in = np.array(y)
        
        # 1. 处理多项式特征 (如果 degree > 1)
        if self.degree > 1:
            self.poly = PolynomialFeatures(degree=self.degree)
            X_transformed = self.poly.fit_transform(X_in)
        else:
            X_transformed = X_in
            # statsmodels 需要手动添加截距项 (Constant/Intercept)
            X_transformed = sm.add_constant(X_transformed)
            
        # 2. 最小二乘法 (OLS) 拟合
        self.model = sm.OLS(y_in, X_transformed)
        self.results = self.model.fit()
        
        # 3. 打印详细的统计报告 (直接截图放论文附录)
        print("-" * 30)
        print(f"回归分析报告 (阶数={self.degree}):")
        print(self.results.summary())
        print("-" * 30)

    def predict(self, X_new):
        """
        预测
        """
        X_new = np.array(X_new)
        if X_new.ndim == 1:
            X_new = X_new.reshape(-1, 1)
            
        if self.degree > 1:
            X_transformed = self.poly.transform(X_new)
        else:
            X_transformed = sm.add_constant(X_new, has_constant='add')
            
        return self.results.predict(X_transformed)

    def plot(self, X, y):
        """
        绘图展示 (仅支持单变量 X 的可视化，多变量X画不出图)
        """
        X_arr = np.array(X)
        if X_arr.ndim > 1 and X_arr.shape[1] > 1:
            print("提示: 自变量多于1个，无法绘制二维拟合曲线图，仅绘制 真实值 vs 预测值 图。")
            y_pred = self.predict(X)
            plt.figure(figsize=(8, 6))
            plt.scatter(y, y_pred, color='blue')
            plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
            plt.xlabel('真实值')
            plt.ylabel('预测值')
            plt.title('真实值 vs 预测值')
            plt.grid(True)
            plt.show()
            return

        # 单变量绘图逻辑
        plt.figure(figsize=(10, 6))
        plt.scatter(X, y, color='black', label='真实数据')
        
        # 生成平滑曲线
        X_range = np.linspace(X_arr.min(), X_arr.max(), 100).reshape(-1, 1)
        y_pred_smooth = self.predict(X_range)
        
        plt.plot(X_range, y_pred_smooth, color='red', linewidth=2, label=f'{self.degree}阶拟合曲线')
        
        # 绘制置信区间 (Confidence Interval) - 高级功能
        # 获取预测结果的详细信息
        if self.degree == 1: # 仅线性回归演示置信区间绘制
            X_smooth_const = sm.add_constant(X_range)
            predictions = self.results.get_prediction(X_smooth_const)
            pred_df = predictions.summary_frame(alpha=0.05) # 95% CI
            
            plt.fill_between(X_range.flatten(), 
                             pred_df['mean_ci_lower'], 
                             pred_df['mean_ci_upper'], 
                             color='red', alpha=0.1, label='95%置信区间')
        
        plt.title(f'回归分析拟合 (Degree={self.degree})')
        plt.xlabel('自变量 X')
        plt.ylabel('因变量 Y')
        plt.legend()
        plt.grid(True)
        plt.show()

# ================= 使用示例 =================

if __name__ == "__main__":
    # 场景1：简单的线性关系 (广告费 -> 销售额)
    print("=== 案例1: 线性回归 ===")
    x1 = [10, 20, 30, 40, 50, 60, 70, 80]
    y1 = [25, 45, 60, 85, 110, 135, 150, 180]
    
    model_lin = RegressionPredictor(polynomial_degree=1)
    model_lin.fit(x1, y1)
    model_lin.plot(x1, y1)
    
    # 场景2：非线性关系 (如: 施肥量 -> 产量，到了某个点会饱和下降)
    print("\n=== 案例2: 二次多项式回归 ===")
    x2 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    # 构造一个先升后降的数据
    y2 = [10, 25, 45, 60, 70, 75, 72, 60, 40, 15] 
    
    model_poly = RegressionPredictor(polynomial_degree=2)
    model_poly.fit(x2, y2)
    
    # 预测一下投入为 5.5 时的情况
    pred_val = model_poly.predict([5.5])
    print(f"输入 5.5，预测输出: {pred_val[0]:.2f}")
    
    model_poly.plot(x2, y2)
```

### 代码使用的“修修补补”指南：

1.  **关于 `polynomial_degree` (阶数)**：
    *   **1 (线性)**：最常用。如果你的散点图大致是一条直线，就选 1。
    *   **2 (二次)**：如果散点图是弯曲的（抛物线状），比如先增后减，或者加速增长，选 2。
    *   *注意*：一般不要超过 3。高阶多项式容易产生**过拟合 (Overfitting)**，即在训练数据上很完美，预测未来时飞到天上去。

2.  **如何解读输出报告 (`results.summary()`)？**
    *   **R-squared ($R^2$)**：拟合优度。越接近1越好。如果是 0.9 以上，论文里可以吹一波“模型拟合效果极佳”。
    *   **P>|t| (P值)**：这是看每个变量是否显著。
        *   找到对应的变量行（如 `x1`, `x2`）。
        *   如果 **P < 0.05**，说明该变量对结果有显著影响（保留）。
        *   如果 **P > 0.05**，说明该变量可能是凑数的，建议从 $X$ 中删掉这一列，重新训练。

3.  **数据需要对数变换吗？**
    *   如果你发现数据差距极大（比如有的数是10，有的数是1个亿），或者散点图呈现指数级爆发。
    *   **操作**：在传入 `fit` 之前，先对 $Y$ 取对数：`y_log = np.log(y)`。
    *   训练完后，预测出来的结果是 $\ln(Y)$，记得用 `np.exp(pred)` 还原回来。
    *   这叫 **对数线性模型**，能有效解决异方差问题。

4.  **多元回归怎么画图？**
    *   如果有多个自变量 ($x_1, x_2, ...$)，你就画不出二维曲线图了。
    *   这时候通常画 **“真实值 vs 预测值”** 的散点图（代码里已经处理了这种情况）。如果点都分布在 $y=x$ 对角线附近，说明预测很准。